# 12 - Predict Single Image (Quick Test)
Test dự đoán nhanh với 1 ảnh bất kỳ — trước khi tích hợp vào app

In [1]:
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time
from ultralytics import YOLO

In [2]:
# ══════════════════════════════════════════════════════════
# 
# ══════════════════════════════════════════════════════════
IMAGE_PATH = "../data/split/test/brown_spot/your_image.jpg"
#                ^^^ Thay bằng đường dẫn ảnh thực tế

# Model paths
YOLO_PATH      = "../models/yolov8n_rice_leaf.pt"
MOBILENET_PATH = "../models/rice_disease_classifier.h5"

# Hyperparams
CONF_THRESHOLD = 0.3
PADDING        = 10

CLASS_NAMES = [
    "bacterial_blight",
    "barrow_brown_leaf_spot",
    "brown_spot",
    "healthy",
    "leaf_blast",
    "leaf_scald",
    "leaf_smut",
    # "neck_blast",
    "rice_hispa",
    "sheath_blight",
    "tungro"
]

CLASS_INFO = {
    "bacterial_blight"    : {"vi": "Bạc lá vi khuẩn",      "severity": "Cao",   "color": "#e74c3c"},
    "barrow_brown_leaf_spot": {"vi": "Đốm nâu Barrow",     "severity": "Trung bình", "color": "#e67e22"},
    "brown_spot"          : {"vi": "Đốm nâu",               "severity": "Trung bình", "color": "#e67e22"},
    "healthy"             : {"vi": "Lá khỏe mạnh",          "severity": "Không", "color": "#27ae60"},
    "leaf_blast"          : {"vi": "Đạo ôn lá",             "severity": "Cao",   "color": "#e74c3c"},
    "leaf_scald"          : {"vi": "Cháy bìa lá",           "severity": "Trung bình", "color": "#e67e22"},
    "leaf_smut"           : {"vi": "Nấm lá",                "severity": "Thấp",  "color": "#f1c40f"},
    # "neck_blast"          : {"vi": "Đạo ôn cổ bông",        "severity": "Rất cao", "color": "#c0392b"},
    "rice_hispa"          : {"vi": "Bọ trĩ lúa (Hispa)",   "severity": "Trung bình", "color": "#e67e22"},
    "sheath_blight"       : {"vi": "Khô vằn",               "severity": "Cao",   "color": "#e74c3c"},
    "tungro"              : {"vi": "Vàng lùn Tungro",       "severity": "Rất cao", "color": "#c0392b"},
}

In [3]:
# ── Load models (chạy 1 lần) ──────────────────────────────────────────
yolo      = YOLO(YOLO_PATH)
mobilenet = tf.keras.models.load_model(MOBILENET_PATH)
print(" Models loaded")

 Models loaded


In [4]:
# ── Inference ─────────────────────────────────────────────────────────
img = cv2.imread(IMAGE_PATH)
assert img is not None, f"❌ Cannot read image: {IMAGE_PATH}"

h_orig, w_orig = img.shape[:2]
t_start = time.perf_counter()

# Step 1: YOLO detect
t0 = time.perf_counter()
yolo_results = yolo.predict(img, conf=CONF_THRESHOLD, verbose=False)
t_yolo = (time.perf_counter() - t0) * 1000

boxes = yolo_results[0].boxes
box_xyxy = None

if boxes is not None and len(boxes) > 0:
    xyxy  = boxes.xyxy.cpu().numpy()
    confs = boxes.conf.cpu().numpy()
    areas = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
    best  = np.argmax(areas)
    x1, y1, x2, y2 = xyxy[best].astype(int)
    x1 = max(0, x1 - PADDING);  y1 = max(0, y1 - PADDING)
    x2 = min(w_orig, x2 + PADDING); y2 = min(h_orig, y2 + PADDING)
    box_xyxy = (x1, y1, x2, y2)
    crop = img[y1:y2, x1:x2]
    yolo_conf = float(confs[best])
    yolo_detected = True
else:
    crop = img.copy()
    yolo_conf = 0.0
    yolo_detected = False

# Step 2: Resize crop
crop_224 = cv2.resize(crop, (224, 224))

# Step 3: MobileNet classify
t1 = time.perf_counter()
img_rgb = cv2.cvtColor(crop_224, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
probs   = mobilenet.predict(np.expand_dims(img_rgb, 0), verbose=0)[0]
t_cls   = (time.perf_counter() - t1) * 1000

pred_idx   = int(np.argmax(probs))
pred_class = CLASS_NAMES[pred_idx]
pred_conf  = float(probs[pred_idx])
t_total    = (time.perf_counter() - t_start) * 1000

info = CLASS_INFO[pred_class]

print("\n" + "═"*55)
print("  PREDICTION RESULT")
print("═"*55)
print(f"  Disease (EN)   : {pred_class}")
print(f"  Disease (VI)   : {info['vi']}")
print(f"  Confidence     : {pred_conf*100:.1f}%")
print(f"  Severity       : {info['severity']}")
print(f"  YOLO detected  : {'Yes (' + str(round(yolo_conf*100)) + '%)' if yolo_detected else 'No (fallback)'}")
print(f"  Latency YOLO   : {t_yolo:.0f} ms")
print(f"  Latency MNet   : {t_cls:.0f} ms")
print(f"  Total latency  : {t_total:.0f} ms")
print("═"*55)

FileNotFoundError: [Errno 2] No such file or directory: '../data/split/test/brown_spot/your_image.jpg'

In [ ]:
# ── Visualize ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 5))

# [A] Ảnh gốc + bbox
ax1 = fig.add_subplot(1, 3, 1)
img_rgb_orig = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
ax1.imshow(img_rgb_orig)
if box_xyxy:
    x1, y1, x2, y2 = box_xyxy
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                               linewidth=3, edgecolor='#00ff88', facecolor='none')
    ax1.add_patch(rect)
    ax1.text(x1, max(y1-8, 0),
             f"rice_leaf  {yolo_conf*100:.0f}%",
             color='white', fontsize=9, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.3', fc='#00aa55', ec='none', alpha=0.85))
det_label = f"YOLO: {'detected' if yolo_detected else 'fallback'}"
ax1.set_title(f'① Original Image\n{det_label}', fontsize=10)
ax1.axis('off')

# [B] Ảnh crop 224×224
ax2 = fig.add_subplot(1, 3, 2)
crop_rgb = cv2.cvtColor(crop_224, cv2.COLOR_BGR2RGB)
ax2.imshow(crop_rgb)
ax2.set_title('② Cropped (224×224)\n→ MobileNet input', fontsize=10)
ax2.axis('off')

# [C] Probability bar chart (top 5)
ax3 = fig.add_subplot(1, 3, 3)
top5_idx  = np.argsort(probs)[::-1][:5]
top5_probs = probs[top5_idx]
top5_names = [CLASS_NAMES[i] for i in top5_idx]
bar_colors = [info['color'] if n == pred_class else '#95a5a6'
              for n in top5_names]

bars = ax3.barh(top5_names[::-1], top5_probs[::-1] * 100, color=bar_colors[::-1])
ax3.set_xlabel('Probability (%)', fontsize=9)
ax3.set_title('③ Top-5 Predictions', fontsize=10)
ax3.set_xlim(0, 105)
for bar, prob in zip(bars, top5_probs[::-1]):
    ax3.text(prob*100 + 0.5, bar.get_y() + bar.get_height()/2,
             f'{prob*100:.1f}%', va='center', fontsize=8)

result_text = f"{pred_class}  ({pred_conf*100:.1f}%)\n{info['vi']}"
plt.suptitle(result_text, fontsize=13, fontweight='bold',
             color=info['color'])
plt.tight_layout()
plt.savefig('prediction_result.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Saved → prediction_result.png")

In [ ]:
# ── Xử lý nhiều ảnh cùng lúc (optional) ──────────────────────────────
def predict_image(image_path):
    """
    Wrapper để gọi từ app hoặc script khác.
    Returns dict: {class_en, class_vi, confidence, severity, yolo_detected}
    """
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Cannot read: {image_path}")
    
    h, w = img.shape[:2]
    
    # YOLO
    res  = yolo.predict(img, conf=CONF_THRESHOLD, verbose=False)
    bxs  = res[0].boxes
    detected = False
    
    if bxs is not None and len(bxs) > 0:
        xyxy  = bxs.xyxy.cpu().numpy()
        areas = (xyxy[:, 2]-xyxy[:, 0]) * (xyxy[:, 3]-xyxy[:, 1])
        x1,y1,x2,y2 = xyxy[np.argmax(areas)].astype(int)
        x1=max(0,x1-PADDING); y1=max(0,y1-PADDING)
        x2=min(w,x2+PADDING); y2=min(h,y2+PADDING)
        crop = img[y1:y2, x1:x2]
        detected = True
    else:
        crop = img
    
    # Classify
    crop = cv2.resize(crop, (224, 224))
    arr  = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    prbs = mobilenet.predict(np.expand_dims(arr, 0), verbose=0)[0]
    idx  = int(np.argmax(prbs))
    cls  = CLASS_NAMES[idx]
    
    return {
        "class_en"      : cls,
        "class_vi"      : CLASS_INFO[cls]["vi"],
        "confidence"    : round(float(prbs[idx]) * 100, 2),
        "severity"      : CLASS_INFO[cls]["severity"],
        "yolo_detected" : detected,
        "all_probs"     : {CLASS_NAMES[i]: round(float(prbs[i])*100, 2)
                           for i in np.argsort(prbs)[::-1][:5]}
    }

# Test
out = predict_image(IMAGE_PATH)
import json
print(json.dumps(out, ensure_ascii=False, indent=2))